# Create intents.json

In [ ]:
import json

intents = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["hi", "hello"],
            "responses": ["Hello!", "Hi"]
        },
        {
            "tag": "bye",
            "patterns": ["bye"],
            "responses": ["Goodbye!"]
        }
    ]
}

with open("intents.json", "w") as f:
    json.dump(intents, f, indent=4)

print("intents.json created")


# Train Chatbot

In [ ]:
import json
import pickle
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

with open("intents.json", "r") as f:
    data = json.load(f)

texts = []
labels = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=2)

classes = sorted(list(set(labels)))
y = np.array([classes.index(label) for label in labels])

model = Sequential([
    Embedding(100, 10),
    LSTM(10),
    Dense(len(classes), activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.fit(X, y, epochs=200, verbose=1)

model.save("chatbot_model.h5")
pickle.dump(tokenizer, open("tokenizer.pkl", "wb"))
pickle.dump(classes, open("classes.pkl", "wb"))

print("Model trained successfully")


# Run Chatbot

In [ ]:
import json
import pickle
import random
import numpy as np

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

model = load_model("chatbot_model.h5")

tokenizer = pickle.load(open("tokenizer.pkl", "rb"))
classes = pickle.load(open("classes.pkl", "rb"))

with open("intents.json", "r") as f:
    data = json.load(f)

print("Chatbot Started (type 'quit' to exit)")

while True:
    msg = input("You: ")

    if msg.lower() == "quit":
        break

    seq = tokenizer.texts_to_sequences([msg])
    seq = pad_sequences(seq, maxlen=2)

    result = model.predict(seq, verbose=0)
    tag = classes[np.argmax(result)]

    for intent in data["intents"]:
        if intent["tag"] == tag:
            print("Bot:", random.choice(intent["responses"]))
            break
